# Build parallel dataset

In [ ]:

from concurrent.futures import ThreadPoolExecutor, as_completed
import os
import time

from pyspark.sql import SparkSession
from pyspark.sql.functions import expr, col, date_format
from pyspark.sql.types import TimestampType

from src.utils import (
    create_df,
    merge_dfs,
    filter_events_by_time_window,
    build_numeric_feature_table,
    build_event_occurrence_feature_table,
)

from config.config import MIMIC_DIR, DATASET_DIR
from config.map_auto import (
    CHARTEVENTS_FEATURES_MAP,
    DATETIMEEVENTS_FEATURES_MAP,
    LABEVENTS_FEATURES_MAP,
    OUTPUTEVENTS_FEATURES_MAP,
    INPUTEVENTS_MV_FEATURES_MAP,
)


spark = SparkSession.builder \
    .appName("MIMIC_III_LengthOfStay_Project") \
    .config("spark.sql.session.timeZone", "UTC") \
    .config("spark.driver.memory", "8g") \
    .config("spark.executor.memory", "8g") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.debug.maxToStringFields", 100) \
    .getOrCreate()

print(f"Sessão Spark iniciada! Versão: {spark.version}")

start = time.time()

Spark Session Initialized! Version: 4.1.1
Sessão Spark iniciada! Versão: 4.1.1


# Base dataset

In [2]:
patients_df = create_df(
    f"{MIMIC_DIR}/PATIENTS.csv",
    columns=["SUBJECT_ID", "GENDER", "DOB", "EXPIRE_FLAG"]
)

admissions_df = create_df(
    f"{MIMIC_DIR}/ADMISSIONS.csv",
    columns=["SUBJECT_ID", "HADM_ID", "ADMISSION_TYPE", "DIAGNOSIS", "ADMITTIME"],
    describe=False
)

icustays_df = create_df(
    f"{MIMIC_DIR}/ICUSTAYS.csv",
    columns=["HADM_ID", "ICUSTAY_ID", "INTIME", "LOS"]
)

merged_df = merge_dfs(patients_df, admissions_df, column="SUBJECT_ID")
merged_df = merge_dfs(merged_df, icustays_df, column="HADM_ID")

merged_df = merged_df.withColumn(
    "END_FIRST_24H",
    expr("INTIME + INTERVAL 24 HOURS")
)

time_reference_df = merged_df.select(
    "SUBJECT_ID",
    "ICUSTAY_ID",
    "ADMITTIME",
    "INTIME",
    "END_FIRST_24H"
).dropDuplicates(["ICUSTAY_ID"]).cache()

time_reference_df.count()


61533

## Feature builders

In [ ]:
def build_chartevents_features():
    events_df = create_df(
        f"{MIMIC_DIR}/chartevents_first_24h.csv",
        columns=["ICUSTAY_ID", "ITEMID", "CHARTTIME", "VALUENUM"],
        describe=False
    )

    events_window = filter_events_by_time_window(
        events_df=events_df,
        reference_df=time_reference_df
    )

    return build_numeric_feature_table(
        events_df=events_window,
        features_map=CHARTEVENTS_FEATURES_MAP,
        id_column="ICUSTAY_ID",
        code_column="ITEMID",
        value_column="VALUENUM",
        metrics=("min", "avg", "max")
    )


def build_datetimeevents_features():
    events_df = create_df(
        f"{MIMIC_DIR}/DATETIMEEVENTS.csv",
        describe=False
    )

    events_window = filter_events_by_time_window(
        events_df=events_df,
        reference_df=time_reference_df,
        join_key="ICUSTAY_ID",
        event_time_col="CHARTTIME",
        start_col="INTIME",
        end_col="END_FIRST_24H",
        value_col="VALUE",
        drop_null_values=True
    )

    return build_event_occurrence_feature_table(
        events_df=events_window,
        features_map=DATETIMEEVENTS_FEATURES_MAP,
        id_column="ICUSTAY_ID",
        code_column="ITEMID"
    )


def build_labevents_features():
    events_df = create_df(
        f"{MIMIC_DIR}/LABEVENTS.csv",
        describe=False
    )

    events_window = filter_events_by_time_window(
        events_df=events_df,
        reference_df=time_reference_df,
        join_key="SUBJECT_ID",
        event_time_col="CHARTTIME",
        start_col="ADMITTIME",
        end_col="END_FIRST_24H",
        value_col="VALUENUM",
        drop_null_values=True
    )

    return build_numeric_feature_table(
        events_df=events_window,
        features_map=LABEVENTS_FEATURES_MAP,
        id_column="ICUSTAY_ID",
        code_column="ITEMID",
        value_column="VALUENUM",
        time_column="CHARTTIME",
        metrics=("latest",)
    )


def build_outputevents_features():
    events_df = create_df(
        f"{MIMIC_DIR}/OUTPUTEVENTS.csv",
        describe=False
    )

    events_window = filter_events_by_time_window(
        events_df=events_df,
        reference_df=time_reference_df,
        join_key="ICUSTAY_ID",
        event_time_col="CHARTTIME",
        start_col="INTIME",
        end_col="END_FIRST_24H",
        value_col="VALUE",
        drop_null_values=True
    )

    return build_numeric_feature_table(
        events_df=events_window,
        features_map=OUTPUTEVENTS_FEATURES_MAP,
        id_column="ICUSTAY_ID",
        code_column="ITEMID",
        value_column="VALUE",
        time_column="CHARTTIME",
        metrics=("sum", "count")
    )


def build_inputevents_mv_features():
    events_df = create_df(
        f"{MIMIC_DIR}/INPUTEVENTS_MV.csv",
        describe=False
    )

    events_window = filter_events_by_time_window(
        events_df=events_df,
        reference_df=time_reference_df,
        join_key="ICUSTAY_ID",
        event_time_col="STARTTIME",
        start_col="INTIME",
        end_col="END_FIRST_24H",
        value_col="AMOUNT",
        drop_null_values=True
    )

    return build_numeric_feature_table(
        events_df=events_window,
        features_map=INPUTEVENTS_MV_FEATURES_MAP,
        id_column="ICUSTAY_ID",
        code_column="ITEMID",
        value_column="AMOUNT",
        time_column="STARTTIME",
        metrics=("sum", "count")
    )

## Parallel execution

In [4]:
feature_jobs = {
    "chartevents": build_chartevents_features,
    "datetimeevents": build_datetimeevents_features,
    "labevents": build_labevents_features,
    "outputevents": build_outputevents_features,
    "inputevents_mv": build_inputevents_mv_features,
}

feature_tables = {}

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = {
        executor.submit(job_function): job_name
        for job_name, job_function in feature_jobs.items()
    }

    for future in as_completed(futures):
        job_name = futures[future]
        print(f"Finalizando features de {job_name}...")

        feature_tables[job_name] = future.result().cache()
        feature_tables[job_name].count()

        print(f"Features de {job_name} concluídas.")


Finalizando features de labevents...
Features de labevents concluídas.
Finalizando features de outputevents...
Features de outputevents concluídas.
Finalizando features de datetimeevents...
Features de datetimeevents concluídas.
Finalizando features de inputevents_mv...
Features de inputevents_mv concluídas.
Finalizando features de chartevents...
Features de chartevents concluídas.


## Final merge

In [5]:
for job_name, feature_df in feature_tables.items():
    print(f"Juntando features de {job_name}...")
    merged_df = merge_dfs(merged_df, feature_df, column="ICUSTAY_ID")

Juntando features de labevents...
Juntando features de outputevents...
Juntando features de datetimeevents...
Juntando features de inputevents_mv...
Juntando features de chartevents...


## Save

In [9]:
os.makedirs(DATASET_DIR, exist_ok=True)

timestamp_cols = [
    field.name
    for field in merged_df.schema.fields
    if isinstance(field.dataType, TimestampType)
]

safe_df = merged_df

for column_name in timestamp_cols:
    safe_df = safe_df.withColumn(
        column_name,
        date_format(col(column_name), "yyyy-MM-dd HH:mm:ss")
    )

safe_df.toPandas().to_csv(
    f"{DATASET_DIR}/final.csv",
    index=False
)

print(f"Tempo total: {time.time() - start}")

Tempo total: 656.103741645813


```
CREATE OR REPLACE TABLE `mimic-496818.mimic_dataset.time_reference` AS
SELECT
  ICUSTAY_ID,
  INTIME,
  TIMESTAMP_ADD(INTIME, INTERVAL 24 HOUR) AS END_FIRST_24H
FROM `mimic-496818.mimic_dataset.icustay_table`
WHERE ICUSTAY_ID IS NOT NULL
  AND INTIME IS NOT NULL;
```

```
CREATE OR REPLACE TABLE `mimic-496818.mimic_dataset.chartevents_first_24h` AS
SELECT
  e.ICUSTAY_ID,
  e.ITEMID,
  e.CHARTTIME,
  e.VALUENUM
FROM `mimic-496818.mimic_dataset.chartevents_table` e
INNER JOIN `mimic-496818.mimic_dataset.time_reference` r
  ON e.ICUSTAY_ID = r.ICUSTAY_ID
WHERE e.CHARTTIME >= r.INTIME
  AND e.CHARTTIME <= r.END_FIRST_24H
  AND e.VALUENUM IS NOT NULL;
```